# 6교시. OCR 및 정보 추출 기능 연동

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/leecks1119/document_ai_lecture/blob/document_ai_lecture_2026/colab/06_ocr_ai_integration.ipynb)

**목표:** 오류를 숨기지 않고 실제·준비 결과 경로를 연결합니다.

**결과물:** `app_06.py`

- 모든 필수 실습은 Google Colab에서 진행합니다.
- API 키나 결제가 필요 없습니다.
- 식별정보를 가린 교육용 샘플 한 장만 사용합니다.
- 개인·회사 문서를 외부 API에 보내거나 공개 Streamlit 주소를 만들지 않습니다.
- 실행이 3분을 넘으면 중지하고 준비 결과를 선택합니다.
- 필요한 셀을 위에서 아래로 다시 실행하고, 계속 실패하면 강사에게 알립니다.
- 실습이 끝나면 Colab 출력과 런타임 파일을 삭제하고 런타임을 종료합니다.


In [ ]:
import platform
import sys
from pathlib import Path

OUTPUT_DIR = Path("course_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("Output:", OUTPUT_DIR.resolve())


In [ ]:
import importlib.metadata
import subprocess

required_streamlit = "1.60.0"
try:
    installed_streamlit = importlib.metadata.version("streamlit")
except importlib.metadata.PackageNotFoundError:
    installed_streamlit = None

if installed_streamlit != required_streamlit:
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            f"streamlit=={required_streamlit}",
        ]
    )

import streamlit
print("Streamlit:", streamlit.__version__)


In [ ]:
SAMPLE_OCR_TEXT = '샘플문구점\n거래일자: 2026-07-27\n연필 2개 × 1,000원 = 2,000원\n노트 1개 × 3,000원 = 3,000원\n합계: 5,000원\n'
SAMPLE_VLM_MARKDOWN = '# 샘플문구점\n\n거래일자: 2026-07-27\n\n| 품목 | 수량 | 단가 | 금액 |\n|---|---:|---:|---:|\n| 연필 | 2 | 1,000원 | 2,000원 |\n| 노트 | 1 | 3,000원 | 3,000원 |\n\n**합계: 5,000원**\n'
SAMPLE_RECEIPT = {'document_type': 'receipt', 'store_name': '샘플문구점', 'date': '2026-07-27', 'total_amount': 5000, 'items': [{'name': '연필', 'quantity': 2, 'unit_price': 1000, 'line_total': 2000}, {'name': '노트', 'quantity': 1, 'unit_price': 3000, 'line_total': 3000}], 'source_mode': 'prepared'}


## 핵심 3개

1. 단계를 작은 함수로 나눕니다.
2. 오류와 사용 모드를 화면에 표시합니다.
3. 준비 결과는 직접 선택합니다.


In [ ]:
def validate_upload(file_path):
    if not file_path:
        return ["파일을 선택하세요."]
    return []


def extract_prepared_result(ocr_text):
    data = dict(SAMPLE_RECEIPT)
    data["source_mode"] = "prepared_extraction"
    return data


def process_document(file_path=None, *, processor="ocr", use_sample=False):
    if processor not in ("ocr", "vlm"):
        return {
            "ok": False,
            "status": "입력 오류",
            "errors": ["processor는 ocr 또는 vlm이어야 합니다."],
        }

    if use_sample:
        document_text = (
            SAMPLE_VLM_MARKDOWN if processor == "vlm" else SAMPLE_OCR_TEXT
        )
        status = (
            "준비 결과: PaddleOCR-VL + 추출"
            if processor == "vlm"
            else "준비 결과: PaddleOCR + 추출"
        )
    else:
        errors = validate_upload(file_path)
        if errors:
            return {
                "ok": False,
                "status": "입력 오류",
                "errors": errors,
                "can_continue_with_sample": True,
            }
        return {
            "ok": False,
            "status": "실제 모델 선택 실행 필요",
            "errors": [
                "PaddleOCR 또는 PaddleOCR-VL 선택 실습을 실행하거나 "
                "'샘플로 계속'을 선택하세요."
            ],
            "can_continue_with_sample": True,
        }

    return {
        "ok": True,
        "status": status,
        "document_text": document_text,
        "data": extract_prepared_result(document_text),
    }


## 실습. 오류 경로와 명시적 준비 결과 경로 확인


In [ ]:
error_result = process_document()
assert not error_result["ok"]
assert "data" not in error_result

sample_result = process_document(processor="vlm", use_sample=True)
assert sample_result["ok"]
assert "준비 결과" in sample_result["status"]
assert sample_result["data"]["total_amount"] == 5000

print(error_result["status"], "→ 사용자가 샘플 선택")
print(sample_result["status"], "→ 완료")


In [ ]:
app_code = 'import streamlit as st\n\nSAMPLE_OCR_TEXT = \'샘플문구점\\n거래일자: 2026-07-27\\n연필 2개 × 1,000원 = 2,000원\\n노트 1개 × 3,000원 = 3,000원\\n합계: 5,000원\\n\'\nSAMPLE_VLM_MARKDOWN = \'# 샘플문구점\\n\\n거래일자: 2026-07-27\\n\\n| 품목 | 수량 | 단가 | 금액 |\\n|---|---:|---:|---:|\\n| 연필 | 2 | 1,000원 | 2,000원 |\\n| 노트 | 1 | 3,000원 | 3,000원 |\\n\\n**합계: 5,000원**\\n\'\nSAMPLE_RECEIPT = {\'document_type\': \'receipt\', \'store_name\': \'샘플문구점\', \'date\': \'2026-07-27\', \'total_amount\': 5000, \'items\': [{\'name\': \'연필\', \'quantity\': 2, \'unit_price\': 1000, \'line_total\': 2000}, {\'name\': \'노트\', \'quantity\': 1, \'unit_price\': 3000, \'line_total\': 3000}], \'source_mode\': \'prepared\'}\n\n\ndef process_document(uploaded=None, *, processor="ocr", use_sample=False):\n    if processor not in ("ocr", "vlm"):\n        return {\n            "ok": False,\n            "status": "입력 오류",\n            "errors": ["처리기는 ocr 또는 vlm이어야 합니다."],\n        }\n\n    if use_sample:\n        document_text = (\n            SAMPLE_VLM_MARKDOWN\n            if processor == "vlm"\n            else SAMPLE_OCR_TEXT\n        )\n        data = dict(SAMPLE_RECEIPT)\n        data["source_mode"] = f"prepared_{processor}"\n        return {\n            "ok": True,\n            "status": f"준비 결과: {processor.upper()} + 추출",\n            "document_text": document_text,\n            "data": data,\n        }\n\n    if uploaded is None:\n        return {\n            "ok": False,\n            "status": "입력 오류",\n            "errors": ["파일을 선택하세요."],\n        }\n\n    return {\n        "ok": False,\n        "status": "실제 모델 실행 필요",\n        "errors": [\n            "2교시 OCR 또는 4교시 VLM 선택 셀의 결과를 연결하세요."\n        ],\n    }\n\n\nst.title("영수증 Document AI 연결 앱")\nuploaded = st.file_uploader(\n    "영수증 이미지 또는 PDF 한 장",\n    type=["png", "jpg", "jpeg", "pdf"],\n)\nprocessor = st.radio(\n    "처리기",\n    options=["ocr", "vlm"],\n    horizontal=True,\n)\nleft, right = st.columns(2)\nrun_uploaded = left.button("업로드 처리", key="run_uploaded")\nrun_sample = right.button("샘플로 계속", key="run_sample")\n\nif run_uploaded:\n    st.session_state["result"] = process_document(\n        uploaded,\n        processor=processor,\n    )\nelif run_sample:\n    st.session_state["result"] = process_document(\n        processor=processor,\n        use_sample=True,\n    )\n\nresult = st.session_state.get("result")\nif result:\n    if result["ok"]:\n        st.success(result["status"])\n        st.text_area("판독 원문", result["document_text"])\n        st.json(result["data"])\n    else:\n        st.error(result["status"])\n        for message in result["errors"]:\n            st.write(f"- {message}")\n'
output_path = OUTPUT_DIR / "app_06.py"
output_path.write_text(app_code, encoding="utf-8")
print("저장 완료:", output_path)


In [ ]:
from streamlit.testing.v1 import AppTest

app_test = AppTest.from_file(str(output_path)).run(timeout=20)
assert not app_test.exception
assert app_test.title[0].value == "영수증 Document AI 연결 앱"
assert len(app_test.file_uploader) == 1
assert len(app_test.button) == 2

app_test.button(key="run_sample").click().run(timeout=20)
assert app_test.success
assert "준비 결과" in app_test.success[0].value
print("독립 실행 Streamlit 앱 검사 완료")


## 준비 결과 경로

필수 경로 자체가 명시적인 준비 결과 실습입니다. 실제 모델 오류 뒤에 자동 전환하지 않고
오류를 확인한 뒤 처리기를 골라 `process_document(use_sample=True)`를 실행합니다.


## 확인

- 오류 결과에 관련 없는 JSON이 없는가?
- 준비 결과 상태가 화면에 분명히 표시되는가?
- 사용자가 직접 샘플 경로를 선택했는가?
